##### Import Functions / Packages

In [1]:
import os
import sys

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from inference import abc_function
from prior import UniformPrior
from simulators import simulate_linear_regression_unknown_sigma
from summaries import regression_summary_with_sd
from distance import euclidean_distance
from plots import plot_posterior

##### Establish Simulation Settings

In [2]:
N_DATASETS = 10
EPSILON = 0.5
N_SIMULATIONS = 100000

TRUE_INTERCEPT = 1
TRUE_SLOPE = 2
TRUE_STANDARD_DEVIATION = 0.5

START = 0
END = 1
OBSERVATIONS = 50

##### Set True Theta

In [3]:
true_theta = {"intercept": TRUE_INTERCEPT, "slope": TRUE_SLOPE, "standard_deviation": TRUE_STANDARD_DEVIATION}

##### Set X values

In [4]:
x = np.linspace(START, END, OBSERVATIONS)

##### Set Prior

In [5]:
prior = UniformPrior({"intercept": (-5, 5), "slope": (-5, 5), "standard_deviation": (0, 5)})

##### Define Simulator and Summary Function

In [6]:
def abc_simulator(theta, abc_rng):
    return simulate_linear_regression_unknown_sigma(theta, x, abc_rng)

def abc_summary(y):
    return regression_summary_with_sd(x, y)

##### Run Simulations

In [7]:
# intialize results list
results = []

for dataset_index in range(N_DATASETS):

    # set rng values for reproducability - ensure they are different but run through same ones
    observed_rng = np.random.default_rng(100 + dataset_index)
    abc_rng = np.random.default_rng(1000 + dataset_index)

    # create observed data
    observed_data = simulate_linear_regression_unknown_sigma(
        theta=true_theta,
        x=x,
        rng=observed_rng,
    )

    # run abc simulation
    accepted_parameters, accepted_distances = abc_function(
        prior=prior,
        simulator=abc_simulator,
        observed_data=observed_data,
        summary_fn=abc_summary,
        distance_fn=euclidean_distance,
        epsilon=EPSILON,
        n_simulations=N_SIMULATIONS,
        rng=abc_rng,
    )

    # collect array of accepted parameters of each parameter
    intercept_samples = np.array([sample["intercept"] for sample in accepted_parameters])
    slope_samples = np.array([sample["slope"] for sample in accepted_parameters])
    sd_samples = np.array([sample["standard_deviation"] for sample in accepted_parameters])

    # get confidence intervals
    intercept_interval = np.quantile(
        intercept_samples,
        [0.025, 0.975])

    slope_interval = np.quantile(
        slope_samples,
        [0.025, 0.975])

    sd_interval = np.quantile(
        sd_samples,
        [0.025, 0.975])

    # add dictionary of results to list of results
    results.append({
        "dataset": dataset_index,
        "accepted": len(accepted_parameters),
        "acceptance_rate": len(accepted_parameters) / N_SIMULATIONS,
        "intercept_mean": np.mean(intercept_samples),
        "intercept_lower": intercept_interval[0],
        "intercept_upper": intercept_interval[1],
        "slope_mean": np.mean(slope_samples),
        "slope_lower": slope_interval[0],
        "slope_upper": slope_interval[1],
        "sd_mean": np.mean(sd_samples),
        "sd_lower": sd_interval[0],
        "sd_upper": sd_interval[1],
        
        # add T/F values for true value being in confidence interval
        "intercept_covered": (
        intercept_interval[0]
        <= true_theta["intercept"]
        <= intercept_interval[1]
        ),
        
        "slope_covered": (
            slope_interval[0]
            <= true_theta["slope"]
            <= slope_interval[1]
        ),
        
        "sd_covered": (
            sd_interval[0]
            <= true_theta["standard_deviation"]
            <= sd_interval[1]
        ),
    })

##### Compile Results

In [8]:
results_df = pd.DataFrame(results)

results_df[["dataset", "accepted", "acceptance_rate", "intercept_mean", "intercept_lower", "intercept_upper",
"slope_mean", "slope_lower", "slope_upper", "sd_mean", "sd_lower", "sd_upper"]]

,dataset,accepted,acceptance_rate,intercept_mean,intercept_lower,intercept_upper,slope_mean,slope_lower,slope_upper,sd_mean,sd_lower,sd_upper
0,0,108,0.00108,1.107242,0.572653,1.585185,1.992711,1.216479,2.638789,0.520274,0.123878,0.936460
1,1,104,0.00104,0.958699,0.344396,1.569840,2.078576,1.257276,2.841724,0.600465,0.201064,1.009891
2,2,112,0.00112,1.236723,0.721888,1.786807,1.664888,0.964998,2.461066,0.635970,0.236566,1.123509
3,3,98,0.00098,0.791857,0.259297,1.360465,2.472536,1.904161,3.143084,0.528185,0.091084,0.997658
4,4,119,0.00119,1.209998,0.635388,1.781646,1.751477,1.005213,2.315834,0.474408,0.049729,0.902392
5,5,102,0.00102,1.112479,0.628264,1.565493,1.776386,0.958148,2.546884,0.575567,0.159670,0.973744
6,6,116,0.00116,1.110437,0.613932,1.568554,1.689812,1.039770,2.387346,0.518222,0.074640,0.950638
7,7,89,0.00089,0.833926,0.248780,1.383422,2.290028,1.550027,3.042050,0.556181,0.141264,1.004251
8,8,108,0.00108,1.331907,0.819442,1.862243,1.606504,0.927032,2.387815,0.633560,0.193700,1.183218
9,9,109,0.00109,1.093354,0.652278,1.531771,1.988070,1.267263,2.595328,0.417723,0.024577,0.850401


In [9]:
coverage = results_df[["intercept_covered", "slope_covered", "sd_covered"]].mean()

coverage

intercept_covered    1.0
slope_covered        1.0
sd_covered           1.0
dtype: float64